# 🎓 Student Exam Score Prediction

Predicting a student's exam score from their daily habits using **Linear Regression**.

**Contents**

1. [Load the data](#1)
2. [Explore the data](#2)
3. [Choose the features](#3)
4. [Train the model](#4)
5. [Evaluate the model](#5)
6. [Understand what the model learned](#6)
7. [Why not Random Forest?](#7)
8. [Save the model](#8)

In [ ]:
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4.5)

RANDOM_STATE = 42  # fixed so every run gives the same result

<a id="1"></a>
## 1. Load the data

In [ ]:
df = pd.read_csv("student_habits_performance.csv")
print(f"{df.shape[0]} students, {df.shape[1]} columns")
df.head()

In [ ]:
df.info()

### Data quality check

Before anything else: are there missing values or duplicated rows?

In [ ]:
missing = df.isna().sum()
print("Missing values:")
print(missing[missing > 0] if missing.any() else "  none")
print(f"\nDuplicate rows: {df.duplicated().sum()}")

Only `parental_education_level` has gaps. As we will see below, that column does
not end up in the model, so no imputation is needed.

In [ ]:
df.describe().T

<a id="2"></a>
## 2. Explore the data

### What does the target look like?

In [ ]:
sns.histplot(df["exam_score"], bins=30, kde=True, color="steelblue")
plt.title("Distribution of exam scores")
plt.xlabel("Exam score")
plt.show()

print(df["exam_score"].describe().round(2).to_string())

Roughly bell-shaped and centred near 70, with no strange spikes. Good for a
regression model.

### Which numeric columns relate to the score?

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation between numeric columns")
plt.show()

In [ ]:
target_corr = (
    corr["exam_score"].drop("exam_score").sort_values(key=abs, ascending=False)
)

colors = ["seagreen" if v > 0 else "indianred" for v in target_corr]
target_corr.plot.barh(color=colors)
plt.gca().invert_yaxis()
plt.title("Correlation with exam score")
plt.xlabel("Correlation")
plt.axvline(0, color="black", linewidth=0.8)
plt.show()

target_corr.round(3)

**Study hours dominate.** Social media and Netflix hours pull the score down,
while sleep, mental health and exercise push it up. `age` is essentially noise.

In [ ]:
top_features = target_corr.abs().nlargest(4).index

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, col in zip(axes.flat, top_features):
    sns.regplot(
        data=df,
        x=col,
        y="exam_score",
        ax=ax,
        scatter_kws={"alpha": 0.3, "s": 12},
        line_kws={"color": "crimson"},
    )
    ax.set_title(f"{col} vs exam score")
plt.tight_layout()
plt.show()

Each relationship follows a straight line. That is the clearest possible signal
that **Linear Regression is the right tool** for this dataset.

### Do the category columns matter?

In [ ]:
categorical_cols = [
    "gender",
    "part_time_job",
    "diet_quality",
    "parental_education_level",
    "internet_quality",
    "extracurricular_participation",
]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.flat, categorical_cols):
    sns.boxplot(data=df, x=col, y="exam_score", ax=ax, hue=col, legend=False)
    ax.set_title(col)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

Every box sits at nearly the same height, so none of these categories separate
high scorers from low scorers. We will confirm this with a real test in the next
section rather than trusting the eye.

<a id="3"></a>
## 3. Choose the features

Rather than guessing, train the same model on three different feature sets and
compare cross-validated scores.

In [ ]:
habit_features = [
    "study_hours_per_day",
    "social_media_hours",
    "netflix_hours",
    "attendance_percentage",
    "sleep_hours",
    "exercise_frequency",
    "mental_health_rating",
]

candidate_sets = {
    "7 habit columns": habit_features,
    "habits + age": habit_features + ["age"],
    "everything (one-hot encoded)": None,  # built below
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
y_all = df["exam_score"]

rows = []
for name, cols in candidate_sets.items():
    if cols is None:
        X_candidate = pd.get_dummies(
            df.drop(columns=["student_id", "exam_score"]), drop_first=True
        ).fillna(0)
    else:
        X_candidate = df[cols]
    score = cross_val_score(
        LinearRegression(), X_candidate, y_all, cv=cv, scoring="r2"
    ).mean()
    rows.append({"feature set": name, "columns": X_candidate.shape[1], "CV R²": score})

pd.DataFrame(rows).sort_values("CV R²", ascending=False).round(4)

The seven habit columns win. Adding age or the encoded categories makes the
model *slightly worse*, because extra columns that carry no signal only add
noise for the model to fit.

**Decision: use the seven habit columns.**

In [ ]:
FEATURES = habit_features
TARGET = "exam_score"

X = df[FEATURES]
y = df[TARGET]

X.head()

<a id="4"></a>
## 4. Train the model

Hold back 20% of students so the model can be scored on data it has never seen.
Scoring on training data would flatter the result.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Training on {len(X_train)} students, testing on {len(X_test)}")

model = LinearRegression()
model.fit(X_train, y_train)

<a id="5"></a>
## 5. Evaluate the model

In [ ]:
y_pred = model.predict(X_test)

rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
mae = float(mean_absolute_error(y_test, y_pred))
r2 = float(r2_score(y_test, y_pred))

print("Scored on the 200 held-out students")
print(f"  R²   : {r2:.4f}   (1.0 would be perfect)")
print(f"  RMSE : {rmse:.3f} points")
print(f"  MAE  : {mae:.3f} points average error")

In [ ]:
# Second opinion: five different train/test rotations.
# If this matches the score above, the result was not a lucky split.
cv_scores = cross_val_score(LinearRegression(), X_train, y_train, cv=cv, scoring="r2")

print("Cross-validated R² per fold:", np.round(cv_scores, 4))
print(f"Mean: {cv_scores.mean():.4f}  (std {cv_scores.std():.4f})")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(y_test, y_pred, alpha=0.5, color="steelblue", edgecolor="none")
lims = [y.min() - 2, y.max() + 2]
ax1.plot(lims, lims, "--", color="crimson", label="perfect prediction")
ax1.set_xlabel("Actual score")
ax1.set_ylabel("Predicted score")
ax1.set_title(f"Predicted vs actual (R² = {r2:.3f})")
ax1.legend()

residuals = y_test - y_pred
ax2.scatter(y_pred, residuals, alpha=0.5, color="seagreen", edgecolor="none")
ax2.axhline(0, color="crimson", linestyle="--")
ax2.set_xlabel("Predicted score")
ax2.set_ylabel("Error (actual − predicted)")
ax2.set_title("Residuals")

plt.tight_layout()
plt.show()

The points hug the diagonal, and the residuals scatter evenly around zero with
no curve or funnel shape. That means a straight-line model is capturing the
pattern properly — there is no leftover structure a fancier model could exploit.

<a id="6"></a>
## 6. Understand what the model learned

This is the advantage of Linear Regression: the entire model is seven numbers.

In [ ]:
coefficients = (
    pd.Series(model.coef_, index=FEATURES)
    .sort_values(key=abs, ascending=False)
    .rename("points per unit")
)

colors = ["seagreen" if v > 0 else "indianred" for v in coefficients]
coefficients.plot.barh(color=colors)
plt.gca().invert_yaxis()
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Effect of each habit on the exam score")
plt.xlabel("Change in exam score per unit")
plt.show()

coefficients.round(3).to_frame()

**Reading this table:** one extra hour of study per day is worth about **+9.5
points**, while an extra hour of social media costs about **−2.7 points**. So one
hour of study outweighs roughly three and a half hours of scrolling.

<a id="7"></a>
## 7. Why not Random Forest?

A fair question for any regression project. Let's actually measure it instead of
assuming.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

comparison = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(
        n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

rows = []
for name, estimator in comparison.items():
    estimator.fit(X_train, y_train)
    preds = estimator.predict(X_test)
    rows.append(
        {
            "model": name,
            "test R²": r2_score(y_test, preds),
            "test RMSE": np.sqrt(mean_squared_error(y_test, preds)),
            "test MAE": mean_absolute_error(y_test, preds),
        }
    )

pd.DataFrame(rows).sort_values("test R²", ascending=False).round(4).reset_index(
    drop=True
)

**Linear Regression wins.** Tree models approximate a straight line with
staircase steps, and with only 1,000 rows they spend their flexibility fitting
noise. Since the relationships here are genuinely linear, the simplest model is
both the most accurate *and* the easiest to explain.

<a id="8"></a>
## 8. Save the model

Now that the honest score is recorded, refit on all 1,000 students so the model
that ships has learned from every available row.

In [ ]:
final_model = LinearRegression()
final_model.fit(X, y)

joblib.dump(final_model, "best_model.pkl")

metrics = {
    "model": "LinearRegression",
    "features": FEATURES,
    "test_r2": r2,
    "test_rmse": rmse,
    "test_mae": mae,
    "cv_r2_mean": float(cv_scores.mean()),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "coefficients": {f: float(c) for f, c in zip(FEATURES, final_model.coef_)},
    "intercept": float(final_model.intercept_),
}

with open("model_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("Saved best_model.pkl and model_metrics.json")

### Try it out

In [ ]:
loaded = joblib.load("best_model.pkl")

example_students = pd.DataFrame(
    [
        {
            "study_hours_per_day": 1.0,
            "social_media_hours": 5.0,
            "netflix_hours": 3.0,
            "attendance_percentage": 60.0,
            "sleep_hours": 5.0,
            "exercise_frequency": 0,
            "mental_health_rating": 3,
        },
        {
            "study_hours_per_day": 3.5,
            "social_media_hours": 2.5,
            "netflix_hours": 1.5,
            "attendance_percentage": 85.0,
            "sleep_hours": 7.0,
            "exercise_frequency": 3,
            "mental_health_rating": 5,
        },
        {
            "study_hours_per_day": 7.0,
            "social_media_hours": 0.5,
            "netflix_hours": 0.5,
            "attendance_percentage": 98.0,
            "sleep_hours": 8.0,
            "exercise_frequency": 6,
            "mental_health_rating": 9,
        },
    ],
    index=["struggling student", "average student", "dedicated student"],
)

# A straight line can run past the ends of the 0-100 scale, so clip it.
example_students["predicted score"] = loaded.predict(example_students).clip(0, 100)
example_students[["study_hours_per_day", "sleep_hours", "predicted score"]].round(1)

---

## Summary

| | |
|---|---|
| **Model** | Linear Regression |
| **Features** | 7 daily-habit columns |
| **R² on unseen data** | ≈ 0.90 |
| **Average error** | ≈ 4.1 points out of 100 |
| **Biggest driver** | Study hours per day (+9.5 points per hour) |

The saved `best_model.pkl` powers the Streamlit app in `app.py`:

```bash
streamlit run app.py
```